# BeverageDzAI — serveur Colab GPU (T4)

Ce notebook lance **Qwen2.5-7B-Instruct-AWQ** avec vLLM et **BAAI/bge-reranker-v2-m3** sur le GPU en FP16. Il crée ensuite une URL Cloudflare temporaire et télécharge le fichier privé `beverage_gpu_connection.env` utilisé par l'application locale.

Avant de commencer : **Exécution → Modifier le type d'exécution → T4 GPU**. Exécutez ensuite les cellules dans l'ordre. La première exécution de la cellule 1 redémarre automatiquement Python après l'installation ; attendez la reconnexion puis relancez uniquement cette cellule avant de continuer.

> L'URL et la clé changent après chaque nouvelle session Colab. Ne partagez jamais le fichier `.env`.

In [ ]:
# 1 — Installation reproductible (la première exécution redémarre Python)
import importlib.metadata as md
import os, subprocess, sys, time

required = {
    "vllm": "0.29.0",
    "torch": "2.13.0",
    "torchvision": "0.28.0",
    "torchaudio": "2.11.0",
}

def installed_version(name):
    try:
        return md.version(name)
    except md.PackageNotFoundError:
        return None

current = {name: installed_version(name) for name in required}
if current != required:
    print("Installation des versions compatibles avec vLLM…")
    subprocess.run(
        [sys.executable, "-m", "pip", "uninstall", "-y",
         "torch", "torchvision", "torchaudio", "torchtext"],
        check=False,
    )
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q",
         "vllm==0.29.0", "sentence-transformers",
         "fastapi", "uvicorn", "httpx", "requests", "jedi"],
        check=True,
    )
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "--force-reinstall",
         "torch==2.13.0", "torchvision==0.28.0", "torchaudio==2.11.0",
         "setuptools>=77.0.3,<81", "numpy<2.5"],
        check=True,
    )
    print("Installation terminée. Reconnexion automatique…")
    time.sleep(3)
    os.kill(os.getpid(), 9)
else:
    print("Dépendances prêtes :", current)

## 2 — Vérifier le GPU et définir les modèles

Sur une T4 de 15 Go, le profil stable est Qwen 7B AWQ avec le reranker sur le même GPU. Le 14B n'est pas activé dans ce notebook, car sa marge VRAM serait insuffisante avec le reranker.

In [ ]:
# 2 — Configuration T4
import os, re, secrets, subprocess, sys, time, requests, torch
from pathlib import Path

if not torch.cuda.is_available():
    raise RuntimeError("GPU absent. Activez Exécution > Modifier le type d'exécution > T4 GPU.")

GPU_NAME = torch.cuda.get_device_name(0)
GPU_GB = torch.cuda.get_device_properties(0).total_memory / 1024**3
MODEL_ID = "Qwen/Qwen2.5-7B-Instruct-AWQ"
SERVED_MODEL = "beverage-qwen"
RERANKER_MODEL = "BAAI/bge-reranker-v2-m3"
RERANKER_DEVICE = "cuda"
MAX_MODEL_LEN = 8192
GPU_UTIL = "0.70"
API_TOKEN = secrets.token_urlsafe(32)

print(f"GPU : {GPU_NAME} ({GPU_GB:.1f} Go)")
print(f"Générateur : {MODEL_ID}")
print(f"Reranker : {RERANKER_MODEL} sur GPU FP16")

## 3 — Démarrer Qwen avec vLLM

Le premier téléchargement peut prendre plusieurs minutes. Attendez le message `vLLM prêt`.

In [ ]:
# 3 — Serveur de génération
if "vllm_process" in globals() and vllm_process.poll() is None:
    vllm_process.terminate()
    vllm_process.wait(timeout=20)

vllm_log = open("/tmp/beverage-vllm.log", "w")
vllm_cmd = [
    sys.executable, "-m", "vllm.entrypoints.openai.api_server",
    "--model", MODEL_ID,
    "--served-model-name", SERVED_MODEL,
    "--host", "127.0.0.1", "--port", "8001",
    "--quantization", "awq",
    "--dtype", "half",
    "--max-model-len", str(MAX_MODEL_LEN),
    "--gpu-memory-utilization", GPU_UTIL,
    "--max-num-seqs", "1",
    "--enforce-eager",
]
vllm_process = subprocess.Popen(vllm_cmd, stdout=vllm_log, stderr=subprocess.STDOUT)
deadline = time.time() + 1800
while time.time() < deadline:
    if vllm_process.poll() is not None:
        print(Path("/tmp/beverage-vllm.log").read_text(errors="replace")[-8000:])
        raise RuntimeError("vLLM s'est arrêté. Le détail utile est affiché ci-dessus.")
    try:
        if requests.get("http://127.0.0.1:8001/v1/models", timeout=5).ok:
            print("vLLM prêt.")
            break
    except requests.RequestException:
        pass
    time.sleep(5)
else:
    raise TimeoutError("vLLM n'a pas démarré dans les 30 minutes.")

## 4 — Démarrer la passerelle et le reranker GPU

La passerelle protège `/v1/chat/completions` et `/rerank` avec une clé Bearer. Si le reranker ne tient exceptionnellement pas en VRAM, il repasse automatiquement sur CPU.

In [ ]:
# 4 — Passerelle authentifiée + BGE reranker
gateway_code = r'''
import os
import httpx
import torch
from fastapi import FastAPI, Request, Response
from pydantic import BaseModel
from sentence_transformers import CrossEncoder

TOKEN = os.environ["BEVERAGE_GPU_TOKEN"]
RERANKER_MODEL = os.environ.get("BEVERAGE_RERANKER_MODEL", "BAAI/bge-reranker-v2-m3")
DEVICE = os.environ.get("BEVERAGE_RERANKER_DEVICE", "cuda")
app = FastAPI(docs_url=None, redoc_url=None, openapi_url=None)

def load_reranker(device):
    kwargs = {"device": device, "max_length": 512}
    if device == "cuda":
        kwargs["model_kwargs"] = {"torch_dtype": torch.float16}
    return CrossEncoder(RERANKER_MODEL, **kwargs)

try:
    reranker = load_reranker(DEVICE)
except RuntimeError as exc:
    if DEVICE != "cuda" or "out of memory" not in str(exc).casefold():
        raise
    print("VRAM insuffisante pour le reranker ; repli CPU.")
    torch.cuda.empty_cache()
    DEVICE = "cpu"
    reranker = load_reranker(DEVICE)

@app.middleware("http")
async def authenticate(request: Request, call_next):
    if request.headers.get("authorization") != f"Bearer {TOKEN}":
        return Response("Unauthorized", status_code=401)
    return await call_next(request)

class RerankRequest(BaseModel):
    model: str
    query: str
    documents: list[str]
    top_n: int = 10

@app.get("/health")
def health():
    return {"status": "ok", "reranker": RERANKER_MODEL, "device": DEVICE}

@app.post("/rerank")
def rerank(payload: RerankRequest):
    if not payload.documents:
        return {"results": []}
    pairs = [(payload.query, doc) for doc in payload.documents]
    try:
        scores = reranker.predict(
            pairs, batch_size=8, show_progress_bar=False,
            activation_fn=torch.nn.Sigmoid(),
        )
    except torch.cuda.OutOfMemoryError:
        torch.cuda.empty_cache()
        scores = reranker.predict(
            pairs, batch_size=2, show_progress_bar=False,
            activation_fn=torch.nn.Sigmoid(),
        )
    order = sorted(range(len(scores)), key=lambda i: float(scores[i]), reverse=True)[:payload.top_n]
    return {"results": [{"index": i, "relevance_score": float(scores[i])} for i in order]}

@app.api_route("/v1/{path:path}", methods=["GET", "POST"])
async def proxy_vllm(path: str, request: Request):
    body = await request.body()
    async with httpx.AsyncClient(timeout=900) as client:
        upstream = await client.request(
            request.method, f"http://127.0.0.1:8001/v1/{path}",
            content=body,
            headers={"content-type": request.headers.get("content-type", "application/json")},
        )
    return Response(
        content=upstream.content, status_code=upstream.status_code,
        media_type=upstream.headers.get("content-type"),
    )
'''
Path("/tmp/beverage_gateway.py").write_text(gateway_code)

if "gateway_process" in globals() and gateway_process.poll() is None:
    gateway_process.terminate()
    gateway_process.wait(timeout=20)

gateway_env = os.environ.copy()
gateway_env["BEVERAGE_GPU_TOKEN"] = API_TOKEN
gateway_env["BEVERAGE_RERANKER_MODEL"] = RERANKER_MODEL
gateway_env["BEVERAGE_RERANKER_DEVICE"] = RERANKER_DEVICE
gateway_log = open("/tmp/beverage-gateway.log", "w")
gateway_process = subprocess.Popen(
    [sys.executable, "-m", "uvicorn", "beverage_gateway:app",
     "--app-dir", "/tmp", "--host", "127.0.0.1", "--port", "8000"],
    env=gateway_env, stdout=gateway_log, stderr=subprocess.STDOUT,
)
headers = {"Authorization": f"Bearer {API_TOKEN}"}
deadline = time.time() + 900
while time.time() < deadline:
    if gateway_process.poll() is not None:
        print(Path("/tmp/beverage-gateway.log").read_text(errors="replace")[-8000:])
        raise RuntimeError("La passerelle s'est arrêtée. Le détail utile est affiché ci-dessus.")
    try:
        response = requests.get("http://127.0.0.1:8000/health", headers=headers, timeout=5)
        if response.ok:
            print("Passerelle et reranker prêts :", response.json())
            break
    except requests.RequestException:
        pass
    time.sleep(5)
else:
    raise TimeoutError("La passerelle n'a pas démarré dans les 15 minutes.")

## 5 — Créer l'URL temporaire et télécharger la connexion

Cette URL change à chaque redémarrage de Colab. Le fichier téléchargé contient l'URL et la clé secrète dont l'application locale a besoin.

In [ ]:
# 5 — Tunnel Cloudflare + fichier de connexion privé
if "tunnel_process" in globals() and tunnel_process.poll() is None:
    tunnel_process.terminate()
    tunnel_process.wait(timeout=20)

cloudflared = Path("/tmp/cloudflared")
if not cloudflared.exists():
    download = requests.get(
        "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64",
        timeout=180,
    )
    download.raise_for_status()
    cloudflared.write_bytes(download.content)
    cloudflared.chmod(0o755)

tunnel_process = subprocess.Popen(
    [str(cloudflared), "tunnel", "--url", "http://127.0.0.1:8000", "--no-autoupdate"],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
)
public_url = None
deadline = time.time() + 120
while time.time() < deadline:
    line = tunnel_process.stdout.readline()
    match = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", line)
    if match:
        public_url = match.group(0)
        break
if not public_url:
    raise RuntimeError("URL Cloudflare introuvable. Relancez cette cellule.")

connection_file = Path("beverage_gpu_connection.env")
connection_file.write_text(
    f"VLLM_BASE_URL={public_url}/v1\n"
    f"RERANKER_BASE_URL={public_url}\n"
    f"VLLM_API_KEY={API_TOKEN}\n"
    f"RERANKER_API_KEY={API_TOKEN}\n"
)
print("URL temporaire créée :", public_url)
try:
    from google.colab import files
    files.download(str(connection_file))
    print("Fichier beverage_gpu_connection.env téléchargé. Gardez-le privé.")
except ImportError:
    print("Fichier créé :", connection_file.resolve())

## 6 — Valider génération et reranking

Cette cellule teste réellement les deux routes publiques avant de déclarer le serveur prêt.

In [ ]:
# 6 — Test de bout en bout
health = requests.get(public_url + "/health", headers=headers, timeout=60)
health.raise_for_status()
rerank_test = requests.post(
    public_url + "/rerank", headers=headers, timeout=180,
    json={
        "model": RERANKER_MODEL,
        "query": "citrus oxidation",
        "documents": ["citral degrades by oxidation", "sugar sweetness"],
        "top_n": 2,
    },
)
rerank_test.raise_for_status()
chat_test = requests.post(
    public_url + "/v1/chat/completions", headers=headers, timeout=300,
    json={
        "model": SERVED_MODEL,
        "messages": [{"role": "user", "content": "Réponds uniquement: OK"}],
        "temperature": 0.0,
        "max_tokens": 8,
    },
)
if not chat_test.ok:
    print(chat_test.text[:4000])
chat_test.raise_for_status()
print("SERVEUR GPU VALIDÉ")
print("État :", health.json())
print("Reranker :", rerank_test.json())
print("Qwen :", chat_test.json()["choices"][0]["message"]["content"])

## 7 — Garder le serveur actif

Lancez cette dernière cellule et laissez l'onglet Colab ouvert pendant l'utilisation de BeverageDzAI. Pour arrêter proprement, cliquez sur le bouton d'arrêt de la cellule.

In [ ]:
# 7 — Maintien de la session
print("Serveur actif. Laissez cette cellule tourner pendant la démo.")
try:
    while all(p.poll() is None for p in [vllm_process, gateway_process, tunnel_process]):
        time.sleep(30)
except KeyboardInterrupt:
    print("Arrêt manuel demandé.")
else:
    stopped = {
        "vLLM": vllm_process.poll(),
        "passerelle": gateway_process.poll(),
        "tunnel": tunnel_process.poll(),
    }
    raise RuntimeError(f"Un service s'est arrêté : {stopped}. Consultez /tmp/beverage-*.log")